# RNA Structure Processing Tutorial

This notebook demonstrates how to use the RNA structure processing pipeline for various tasks.

## Setup

First, let's import the necessary modules and set up our environment.

In [1]:
import sys
sys.path.append('.')

from utils.pdb_downloader import PDBDownloader
from utils.pdb_to_npy import PDBToNumpyConverter
from utils.npy_to_pdb import NumpyToPDBConverter
from utils.extract_loops import LoopExtractor
from utils.extract_rna_segments import RNAExtractor
from utils.extract_sequence import SequenceExtractor
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

## A. Data Acquisition

### Downloading RNA structures from PDB

In [ ]:
# Initialize the downloader
downloader = PDBDownloader()

# Example 1: Download specific PDB IDs
pdb_ids = ['1ABC', '2XYZ']  # Replace with actual PDB IDs
downloader.download_pdbs(pdb_ids)

# Example 2: Search and download by criteria
criteria = {
    'resolution': 3.0,  # Maximum resolution
    'rna_only': True,  # RNA-only structures
    'min_length': 50   # Minimum sequence length
}
downloader.search_and_download(criteria)

In [ ]:
   # Download specific PDB IDs with limit
   !python pdb_downloader.py --pdb-ids 1ABC 2XYZ --max-pdbs 5

   # Download from file with limit
   !python pdb_downloader.py --input-file pdb_list.txt --max-pdbs 10

   # Search and download with limit
   !python pdb_downloader.py --search --rna-only --max-pdbs 20

## B. Structure Processing

### Converting PDB files to NumPy arrays

In [ ]:
# Initialize the converter
converter = PDBToNumpyConverter(
    processed_dir="processed_pdbs",
    npy_dir="npy_files"
)

# Convert all PDB files
converter.convert_all_pdbs()

# Example: Load and visualize a NumPy array
npy_file = "npy_files/1ABC.npy"
if os.path.exists(npy_file):
    data = np.load(npy_file)
    print(f"Array shape: {data.shape}")
    print(f"Number of residues: {data.shape[0]}")
    
    # Plot atom positions for first residue
    plt.figure(figsize=(10, 6))
    plt.scatter(data[0, :, 0], data[0, :, 1])
    plt.title("Atom positions for first residue")
    plt.show()

In [ ]:
!python pdb_to_npy.py --input-dir processed_pdbs --output-dir npy_files

### Reconstructing PDB files from NumPy arrays

In [ ]:
# Initialize the converter
converter = NumpyToPDBConverter(
    npy_dir="npy_files",
    output_dir="reconstructed_pdbs"
)

# Convert all NumPy files
converter.convert_all_npy()

# Example: Convert a specific file
npy_file = "npy_files/1ABC.npy"
fasta_file = "sequences/1ABC.fasta"
if os.path.exists(npy_file) and os.path.exists(fasta_file):
    converter.convert_single_file(npy_file, fasta_file)

In [ ]:
!python scripts/npy_to_pdb.py --npy-dir competition/train/coords --fasta-dir competition/train/seqs --output-dir reconstructed_pdbs

## C. RNA Loop Extraction

### Extracting loop regions

In [2]:
# # Initialize the extractor
# extractor = LoopExtractor(
#     input_dir="reconstructed_pdbs",
#     output_dir="extracted_loops",
#     distance_cutoff=10.0  # 10Å cutoff
# )

# # Process all files
# extractor.process()

# Example: Extract loops with different cutoffs
for cutoff in [8.0, 10.0, 12.0]:
    extractor = LoopExtractor(
        input_dir="reconstructed_pdbs",
        output_dir=f"extracted_loops_{cutoff}",
        distance_cutoff=cutoff
    )
    extractor.process()

2025-05-27 18:52:15,973 - INFO - Found 1981 PDB files to process
2025-05-27 18:52:15,974 - INFO - Processing 9D0J_1_1x.pdb
/Users/xiaojuzhang/opt/anaconda3/lib/python3.9/site-packages/Bio/SeqRecord.py:228: BiopythonDeprecationWarning: Using a string as the sequence is deprecated and will raise a TypeError in future. It has been converted to a Seq object.
  warnings.warn(
2025-05-27 18:52:16,055 - INFO - Processing 4U4O_1_2.pdb
2025-05-27 18:52:16,261 - INFO - Processing 4V5C_1_BA.pdb
2025-05-27 18:52:16,535 - INFO - Processing 1HNW_1_X.pdb
/Users/xiaojuzhang/opt/anaconda3/lib/python3.9/site-packages/Bio/SeqRecord.py:228: BiopythonDeprecationWarning: Using a string as the sequence is deprecated and will raise a TypeError in future. It has been converted to a Seq object.
  warnings.warn(
2025-05-27 18:52:16,538 - INFO - Processing 5L3P_1_x.pdb
/Users/xiaojuzhang/opt/anaconda3/lib/python3.9/site-packages/Bio/SeqRecord.py:228: BiopythonDeprecationWarning: Using a string as the sequence is 

In [ ]:
#!python rna_loop_extractor.py --input-dir reconstructed_pdbs --output-dir extracted_loops --distance-cutoff 10.0

## D. RNA Fragment Extraction

### Extracting random RNA segments

In [ ]:
# Example 1: Extract segments with default coverage rate
extractor = RNAExtractor(
    input_dir="reconstructed_pdbs",
    min_length=3,
    max_length=20,
    coverage_rate=0.02,
    generation_id="run1"
)

# Process all files
extractor.process()

# # Example 2: Extract segments of different lengths with different coverage rates
# length_ranges = [
#     (5, 10, 0.15),   # 15% coverage for short segments
#     (10, 15, 0.1),   # 10% coverage for medium segments
#     (15, 20, 0.05)   # 5% coverage for longer segments
# ]

# for min_len, max_len, coverage in length_ranges:
#     extractor = RNAExtractor(
#         input_dir="processed_pdbs",
#         min_length=min_len,
#         max_length=max_len,
#         coverage_rate=coverage
#     )
#     extractor.process()

In [ ]:
!python scripts/extract_rna_segments.py --input-dir reconstructed_pdbs --generation-id run1 --min-length 5 --max-length 20 --num-extractions 5

## E. Sequence Analysis

### Extracting FASTA sequences

In [ ]:
# Initialize the extractor
extractor = SequenceExtractor(
    pdb_dir="processed_pdbs",
    output_dir="sequences"
)

# Process all files
extractor.process()

# Example: Analyze sequence properties
def analyze_sequence(fasta_file):
    with open(fasta_file, 'r') as f:
        lines = f.readlines()
    
    sequence = ''.join(lines[1:]).strip()
    print(f"Sequence length: {len(sequence)}")
    print(f"Base composition: {dict(zip('AUGC', [sequence.count(b) for b in 'AUGC']))}")

# Analyze a specific file
fasta_file = "sequences/1ABC.fasta"
if os.path.exists(fasta_file):
    analyze_sequence(fasta_file)

In [ ]:
!python scripts/pdb_to_fasta.py --input-dir reconstructed_pdbs --output-dir sequences

## Error Handling and Best Practices

Here are some examples of proper error handling and best practices:

In [ ]:
# Example 1: Safe file operations
def safe_file_operation(file_path):
    try:
        with open(file_path, 'r') as f:
            return f.read()
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return None
    except Exception as e:
        print(f"Error reading file: {e}")
        return None

# Example 2: Batch processing with progress tracking
def process_files(file_list):
    total = len(file_list)
    for i, file in enumerate(file_list, 1):
        print(f"Processing file {i}/{total}: {file}")
        try:
            # Process file
            pass
        except Exception as e:
            print(f"Error processing {file}: {e}")
            continue

# Example 3: Memory-efficient processing
def process_large_file(file_path, chunk_size=1000):
    with open(file_path, 'r') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            # Process chunk
            yield chunk